<a href="https://colab.research.google.com/github/step-code01/Research-Temporal/blob/main/v2.0.0_temporal_probing_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Kaam:** Understand WHERE and WHAT temporal information is encoded in VideoLLMs

**Approach:**
- **WHERE:** Multi-layer probing (which layers encode temporal info?)
- **WHAT:** Interventions (does model use temporal order or shortcuts?)

**Dataset:** SSv2 POC subset (600 videos, 6 classes)

## 1. Setup & Installation

In [ ]:
!pip install -q transformers accelerate torch torchvision decord scikit-learn matplotlib seaborn pandas tqdm

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from transformers import VideoMAEImageProcessor, VideoMAEModel
from google.colab import drive
import decord
from decord import VideoReader, cpu

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Drive & Setup Paths

In [ ]:
drive.mount('/content/drive')

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/Dataset"
VIDEO_DIR = f"{PROJECT_ROOT}/poc_videos"
EMBEDDINGS_DIR = f"{PROJECT_ROOT}/embeddings"
RESULTS_DIR = f"{PROJECT_ROOT}/results"

for dir_path in [PROJECT_ROOT, EMBEDDINGS_DIR, RESULTS_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("Project Setup:")
print(f"Project Root: {PROJECT_ROOT}")
print(f"Videos: {VIDEO_DIR}")
print(f"Embeddings: {EMBEDDINGS_DIR}")
print(f"Results: {RESULTS_DIR}")

## 3. Load Metadata & Verify Data

In [ ]:
metadata_path = f"{VIDEO_DIR}/metadata.json"
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print("Dataset Info:")
print(f"Total videos: {metadata['total_videos']}")
print(f"Videos per class: {metadata['videos_per_class']}")
print(f"\nClasses:")
for class_key, template in metadata['classes'].items():
    num_videos = len(metadata['selected_videos'][class_key])
    print(f"  {class_key:15s}: {num_videos:3d} videos - '{template}'")

In [ ]:
#Creating dataset dataframe
data_rows = []
for class_key, videos in metadata['selected_videos'].items():
    for video in videos:
        video_path = f"{VIDEO_DIR}/{class_key}/{video['id']}.webm"
        data_rows.append({
            'video_id': video['id'],
            'class': class_key,
            'template': video['template'],
            'split': video['split'],
            'path': video_path
        })

df = pd.DataFrame(data_rows)

df['exists'] = df['path'].apply(os.path.exists) #apply function to each column (default axis =0 column)
print(f"\n Videos found: {df['exists'].sum()}/{len(df)}")

if df['exists'].sum() < len(df):
    print(f"\n Missing {(~df['exists']).sum()} videos!")
    print("Please upload the poc_videos folder to Google Drive at:")
    print(f"  {VIDEO_DIR}")
else:
    print("\n All videos found!")

print("\nClass distribution:")
print(df['class'].value_counts())

print("\nSplit distribution:")
print(df.groupby(['split', 'class']).size().unstack(fill_value=0))

df.head()

## 4. Video Loading Utilities

In [ ]:
print(df.iat[0,4])

In [ ]:
def load_video(video_path, num_frames=16):
    """
    Load video and sample frames uniformly.

    Args:
        video_path: Path to video file
        num_frames: Number of frames to sample

    Returns:
        frames: numpy array of shape (num_frames, H, W, C)
    """
    try:
        vr = VideoReader(video_path, ctx=cpu(0), num_threads=1)
        total_frames = len(vr)

        # Sample frames uniformly
        indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
        frames = vr.get_batch(indices).asnumpy()

        return frames
    except Exception as e:
        print(f"Error loading {video_path}: {e}")
        return None

# Test loading
test_video = df[df['exists']].iloc[0]['path'] #df['exists'] returns pandas Series object jispe first row selecting and uska path
test_frames = load_video(test_video)
if test_frames is not None:
    print(f"Loaded test video: {test_frames.shape}")

    # Visualize a few frames
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for i, idx in enumerate([0, 5, 10, 15]): #enumerate, get both index & element at same time
        axes[i].imshow(test_frames[idx])
        axes[i].axis('off')
        axes[i].set_title(f'Frame {idx}')
    plt.tight_layout()
    plt.show()
else:
    print("Failed to load test video")

#decord VideoReader issue (ffmpeg under the hood) need
#to diagnose more robust something

Wait! Threading issue kyu maar rhi thi? single thread pe error eh for decord?

doing num_threads = 1 in VideoReader() solved it.

###### Debugging decord's ffmpeg

In [ ]:
!ffprobe -v error /content/drive/MyDrive/Dataset/poc_videos/horizontal_rl/187588.webm

In [ ]:
!ffmpeg -codecs | grep vp9

In [ ]:
!ffprobe -v error -show_format -show_streams /content/drive/MyDrive/Dataset/poc_videos/horizontal_rl/187588.webm

ctrl f8 is shortcut to run all cells above the current cell

## 5. Load VideoMAE Model

Models Testing:
1. MCG-NJU/videomae-base <br>
2.MCG-NJU/videomae-base-finetuned-kinetics

Exhaustive study need to do:
base models, more/less param models of same model, finetuned version, check all.

In [ ]:
model_name = "MCG-NJU/videomae-base-finetuned-kinetics"

print(f"Loading model: {model_name}")
processor = VideoMAEImageProcessor.from_pretrained(model_name)
model = VideoMAEModel.from_pretrained(model_name,attn_implementation="sdpa",dtype=torch.float16,)
model = model.to(device)
model.eval() #evaluation mode set

print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print(f"Number of layers: {len(model.encoder.layer)}")

## 6. Feature Extraction Function

In [ ]:
def extract_features(video_path, model, processor, layer_indices=None, intervention=None):
    """
    Extract features from specific layers of VideoMAE.

    Args:
        video_path: Path to video
        model: VideoMAE model
        processor: VideoMAE processor
        layer_indices: List of layer indices to extract from (None = all layers)
        intervention: Type of intervention ('reverse', 'shuffle', None)

    Returns:
        features_dict: {layer_idx: feature_tensor}
    """
    frames = load_video(video_path, num_frames=16)
    if frames is None:
        return None

    # Apply intervention
    if intervention == 'reverse':
        frames = frames[::-1]  # Reverse temporal order
    elif intervention == 'shuffle':
        indices = np.random.permutation(len(frames))
        frames = frames[indices]  # Shuffle frames

    # Preprocess
    inputs = processor(list(frames), return_tensors="pt")
    # Cast inputs to the same dtype as the model
    inputs = {k: v.to(device).to(model.dtype) for k, v in inputs.items()}

    # Extract features
    features_dict = {}

    with torch.no_grad():
        # Get all hidden states
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # Tuple of (batch, seq_len, hidden_dim)

        # Extract from specified layers
        if layer_indices is None:
            layer_indices = range(len(hidden_states))

        for layer_idx in layer_indices:
            # Use CLS token (first token) or mean pool
            layer_output = hidden_states[layer_idx]  # (batch, seq_len, hidden_dim)
            # Mean pool over sequence
            feature = layer_output.mean(dim=1).cpu().numpy()  # (batch, hidden_dim)
            features_dict[layer_idx] = feature[0]  # Remove batch dimension

        #FIX: delete intermediate tensors
        del outputs
        del hidden_states
        del layer_output

    #clear gpu cache
    del inputs
    torch.cuda.empty_cache()

    return features_dict

# Test feature extraction
print("Testing feature extraction...")
test_features = extract_features(test_video, model, processor, layer_indices=[0, 6, 11])

if test_features:
    print("\nFeature extraction successful!")
    for layer_idx, feat in test_features.items():
        print(f"Layer {layer_idx:2d}: shape {feat.shape}")
else:
    print("Feature extraction failed")

issue was model loaded in half precision (torch.float16) and data loaded in (torch.float32) so typecast kardiya at line 53 (shape error)

## 7. Extract and Save Embeddings (GPU Phase)

In [ ]:
# Define layers to extract
# VideoMAE-base has 12 layers (0-11)
# We'll sample: 0%, 25%, 50%, 75%, 100% of layers
num_layers = len(model.encoder.layer)
layer_indices = [0, num_layers // 4, num_layers // 2, 3 * num_layers // 4, num_layers - 1]

print(f"Extracting from layers: {layer_indices}")
print(f"Total layers in model: {num_layers}")

In [ ]:
def extract_and_save_embeddings(df, layer_indices, intervention=None, save_dir=None):
    """
    Extract embeddings for all videos and save to disk.
    """
    if save_dir is None:
        intervention_suffix = f"_{intervention}" if intervention else ""
        save_dir = f"{EMBEDDINGS_DIR}/embeddings{intervention_suffix}"

    os.makedirs(save_dir, exist_ok=True)

    # Storage for embeddings
    embeddings = {layer_idx: [] for layer_idx in layer_indices}
    labels = []
    video_ids = []
    splits = []

    print(f"\nExtracting embeddings (intervention={intervention})...")

    #batch processing to periodically save and free memory
    SAVE_EVERY = 50

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
        if not row['exists']:
            continue

        # Extract features
        features = extract_features(
            row['path'],
            model,
            processor,
            layer_indices=layer_indices,
            intervention=intervention
        )

        if features is None:
            continue

        # Store
        for layer_idx in layer_indices:
            embeddings[layer_idx].append(features[layer_idx])

        labels.append(row['class'])
        video_ids.append(row['video_id'])
        splits.append(row['split'])

        #periodic memory cleanup
        if len(labels) % SAVE_EVERY == 0:
          torch.cuda.empty_cache()
          print(f"[Memory cleanup at {len(labels)} videos]")

    # Convert to numpy arrays and save
    print("\nSaving embeddings...")
    for layer_idx in layer_indices:
        embeddings[layer_idx] = np.array(embeddings[layer_idx])

        # Save embeddings
        emb_file = f"{save_dir}/layer_{layer_idx}.npy"
        np.save(emb_file, embeddings[layer_idx])
        print(f"  Saved layer {layer_idx:2d}: {embeddings[layer_idx].shape} -> {emb_file}")

    # Save metadata
    metadata = {
        'labels': labels,
        'video_ids': video_ids,
        'splits': splits,
        'layer_indices': layer_indices,
        'intervention': intervention
    }

    metadata_file = f"{save_dir}/metadata.json"
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"\nSaved metadata: {metadata_file}")
    print(f"Total videos processed: {len(labels)}")

    return embeddings, metadata

issue is it keeps 491 videos * 5 layers each * hidden states = huge memory leak!

even with torch.no_grad() pytorch keeps immediate tensors in gpu until explicitely cleared.

memory isn't being freed between videos, and after 491 videos, 15gb filled!

In [ ]:
# Extract normal embeddings (no intervention)
embeddings_normal, metadata_normal = extract_and_save_embeddings(
    df,
    layer_indices=layer_indices,
    intervention=None
)

~~7B Model OOM kr gya 14GB pe even at half precision?~~ fixed

In [ ]:
# Extract REVERSED embeddings
embeddings_reversed, metadata_reversed = extract_and_save_embeddings(
    df,
    layer_indices=layer_indices,
    intervention='reverse'
)

In [ ]:
# Extract SHUFFLED embeddings
embeddings_shuffled, metadata_shuffled = extract_and_save_embeddings(
    df,
    layer_indices=layer_indices,
    intervention='shuffle'
)

## 8. Train Probes
(CPU Phase, no GPU needed here)

In [ ]:
def load_embeddings(save_dir):
    """
    Load pre-extracted embeddings from disk.
    """
    # Load metadata
    with open(f"{save_dir}/metadata.json", 'r') as f:
        metadata = json.load(f)

    # Load embeddings
    embeddings = {}
    for layer_idx in metadata['layer_indices']:
        emb_file = f"{save_dir}/layer_{layer_idx}.npy"
        embeddings[layer_idx] = np.load(emb_file)

    return embeddings, metadata

# Load all embeddings
print("Loading embeddings...")
embeddings_normal, metadata_normal = load_embeddings(f"{EMBEDDINGS_DIR}/embeddings")
embeddings_reversed, metadata_reversed = load_embeddings(f"{EMBEDDINGS_DIR}/embeddings_reverse")
embeddings_shuffled, metadata_shuffled = load_embeddings(f"{EMBEDDINGS_DIR}/embeddings_shuffle")
print("All embeddings loaded!")

In [ ]:
def train_probe(X, y, probe_type='linear', random_state=42):
    """
    Train a probe (linear or MLP).

    Args:
        X: Features (n_samples, n_features)
        y: Labels (n_samples,)
        probe_type: 'linear' or 'mlp'

    Returns:
        accuracy, model
    """
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state, stratify=y
    )

    # Train probe
    if probe_type == 'linear':
        probe = LogisticRegression(max_iter=1000, random_state=random_state)
    else:  # mlp
        probe = MLPClassifier(
            hidden_layer_sizes=(256, 128),
            max_iter=1000,
            random_state=random_state
        )

    probe.fit(X_train, y_train)

    # Evaluate
    y_pred = probe.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy, probe

# Test probe training
print("Testing probe training...")
test_X = embeddings_normal[0]  # Layer 0
test_y = np.array(metadata_normal['labels'])

acc_linear, _ = train_probe(test_X, test_y, probe_type='linear')
acc_mlp, _ = train_probe(test_X, test_y, probe_type='mlp')

print(f"\nTest results (Layer 0):")
print(f"  Linear probe: {acc_linear:.3f}")
print(f"  MLP probe: {acc_mlp:.3f}")

## 9. Multi-Layer Probing Analysis (WHERE)

hai kahan temporal info?

In [ ]:
def run_multilayer_probing(embeddings, metadata, probe_type='linear'):
    """
    Train probes on all layers.
    """
    results = []
    labels = np.array(metadata['labels'])

    print(f"\nTraining {probe_type} probes on all layers...")

    for layer_idx in metadata['layer_indices']:
        X = embeddings[layer_idx]
        acc, probe = train_probe(X, labels, probe_type=probe_type)

        results.append({
            'layer': layer_idx,
            'accuracy': acc,
            'probe_type': probe_type
        })

        print(f"  Layer {layer_idx:2d}: {acc:.3f}")

    return pd.DataFrame(results)

# Run for both probe types
results_linear = run_multilayer_probing(embeddings_normal, metadata_normal, probe_type='linear')
results_mlp = run_multilayer_probing(embeddings_normal, metadata_normal, probe_type='mlp')

# Combine results
results_df = pd.concat([results_linear, results_mlp], ignore_index=True)
results_df

In [ ]:
# Visualize WHERE temporal info is encoded
fig, ax = plt.subplots(figsize=(10, 6))

for probe_type in ['linear', 'mlp']:
    data = results_df[results_df['probe_type'] == probe_type]
    ax.plot(data['layer'], data['accuracy'], marker='o', label=f'{probe_type.upper()} Probe', linewidth=2)

ax.set_xlabel('Layer Index', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('WHERE: Multi-Layer Probing Results\n(Which layers encode temporal information?)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/multilayer_probing.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved: {RESULTS_DIR}/multilayer_probing.png")

## 10. Intervention Analysis (WHAT)

what is videoLLM using?

In [ ]:
def run_intervention_analysis(embeddings_dict, labels, probe_type='linear'):
    """
    Compare probe performance across interventions.

    Args:
        embeddings_dict: {intervention: {layer: embeddings}}
        labels: Array of labels
        probe_type: 'linear' or 'mlp'
    """
    results = []

    for intervention, embeddings in embeddings_dict.items():
        print(f"\n{intervention.upper()} intervention:")

        for layer_idx in sorted(embeddings.keys()):
            X = embeddings[layer_idx]
            acc, _ = train_probe(X, labels, probe_type=probe_type)

            results.append({
                'intervention': intervention,
                'layer': layer_idx,
                'accuracy': acc,
                'probe_type': probe_type
            })

            print(f"  Layer {layer_idx:2d}: {acc:.3f}")

    return pd.DataFrame(results)

# Prepare data
embeddings_dict = {
    'normal': embeddings_normal,
    'reversed': embeddings_reversed,
    'shuffled': embeddings_shuffled
}
labels = np.array(metadata_normal['labels'])

# Run analysis for both probe types

print("INTERVENTION ANALYSIS - LINEAR PROBES")
intervention_results_linear = run_intervention_analysis(embeddings_dict, labels, probe_type='linear')

print("\nINTERVENTION ANALYSIS - MLP PROBES")
intervention_results_mlp = run_intervention_analysis(embeddings_dict, labels, probe_type='mlp')

# Combine
intervention_results = pd.concat([intervention_results_linear, intervention_results_mlp], ignore_index=True)
intervention_results

In [ ]:
# Visualize WHAT the model uses
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, probe_type in enumerate(['linear', 'mlp']):
    ax = axes[idx]
    data = intervention_results[intervention_results['probe_type'] == probe_type]

    for intervention in ['normal', 'reversed', 'shuffled']:
        subset = data[data['intervention'] == intervention]
        ax.plot(subset['layer'], subset['accuracy'], marker='o', label=intervention.capitalize(), linewidth=2)

    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'{probe_type.upper()} Probe', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

fig.suptitle('WHAT: Intervention Analysis\n(Does the model use temporal order or just frame content?)',
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/intervention_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n Figure saved: {RESULTS_DIR}/intervention_analysis.png")

## 11. Combined Heatmap Visualization

In [ ]:
# Create heatmap showing accuracy drop from interventions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, probe_type in enumerate(['linear', 'mlp']):
    data = intervention_results[intervention_results['probe_type'] == probe_type]

    # Pivot for heatmap
    pivot = data.pivot(index='intervention', columns='layer', values='accuracy')
    pivot = pivot.reindex(['normal', 'reversed', 'shuffled'])  # Order rows

    # Create heatmap
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
                ax=axes[idx], cbar_kws={'label': 'Accuracy'})
    axes[idx].set_title(f'{probe_type.upper()} Probe', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel('Layer Index', fontsize=12)
    axes[idx].set_ylabel('Intervention', fontsize=12)

fig.suptitle('Intervention Impact Heatmap', fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/intervention_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Figure saved: {RESULTS_DIR}/intervention_heatmap.png")

Linear probing uses a single linear layer to test if frozen, pre-trained representations are linearly separable for a task, indicating high-quality, easily accessible information.

MLP probing adds non-linear layers, increasing capacity to detect more complex, abstract, or non-linear relationships within the features, potentially yielding higher accuracy at the risk of higher overfitting

## 12. Compute Accuracy Drops

In [ ]:
# Calculate accuracy drops from interventions
def compute_accuracy_drops(results_df):
    """
    Compute accuracy drop from normal to intervention.
    """
    drops = []

    for probe_type in results_df['probe_type'].unique():
        for layer in results_df['layer'].unique():
            # Get normal accuracy
            normal_acc = results_df[
                (results_df['probe_type'] == probe_type) &
                (results_df['layer'] == layer) &
                (results_df['intervention'] == 'normal')
            ]['accuracy'].values[0]

            # Compute drops
            for intervention in ['reversed', 'shuffled']:
                interv_acc = results_df[
                    (results_df['probe_type'] == probe_type) &
                    (results_df['layer'] == layer) &
                    (results_df['intervention'] == intervention)
                ]['accuracy'].values[0]

                drop = normal_acc - interv_acc
                drop_pct = (drop / normal_acc) * 100 if normal_acc > 0 else 0

                drops.append({
                    'probe_type': probe_type,
                    'layer': layer,
                    'intervention': intervention,
                    'normal_acc': normal_acc,
                    'intervention_acc': interv_acc,
                    'drop_absolute': drop,
                    'drop_percent': drop_pct
                })

    return pd.DataFrame(drops)

accuracy_drops = compute_accuracy_drops(intervention_results)

print("\nAccuracy Drops from Interventions:")
print(accuracy_drops.to_string(index=False))

# Save results
accuracy_drops.to_csv(f"{RESULTS_DIR}/accuracy_drops.csv", index=False)
print(f"\n Saved: {RESULTS_DIR}/accuracy_drops.csv")

In [ ]:
# Visualize accuracy drops
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, intervention in enumerate(['reversed', 'shuffled']):
    ax = axes[idx]
    data = accuracy_drops[accuracy_drops['intervention'] == intervention]

    for probe_type in ['linear', 'mlp']:
        subset = data[data['probe_type'] == probe_type]
        ax.plot(subset['layer'], subset['drop_absolute'], marker='o',
                label=f'{probe_type.upper()}', linewidth=2)

    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Accuracy Drop (Absolute)', fontsize=12)
    ax.set_title(f'{intervention.capitalize()} Intervention', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)

fig.suptitle('Accuracy Drops from Interventions\n(Higher = More reliance on temporal order)',
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/accuracy_drops.png", dpi=150, bbox_inches='tight')
plt.show()

print(f" Figure saved: {RESULTS_DIR}/accuracy_drops.png")

## 13. Summary Report

In [ ]:
# Generate summary report
report = []
report.append("="*80)
report.append("TEMPORAL PROBING POC - SUMMARY REPORT")
report.append("="*80)
report.append("")

# Dataset info
report.append("DATASET:")
report.append(f"  Total videos: {len(labels)}")
report.append(f"  Classes: {len(set(labels))}")
report.append(f"  Class distribution: {dict(pd.Series(labels).value_counts())}")
report.append("")

# Model info
report.append("MODEL:")
report.append(f"  Name: {model_name}")
report.append(f"  Layers analyzed: {layer_indices}")
report.append("")

# WHERE results
report.append("WHERE: MULTI-LAYER PROBING RESULTS")
report.append("-" * 80)
for probe_type in ['linear', 'mlp']:
    data = results_df[results_df['probe_type'] == probe_type]
    best_layer = data.loc[data['accuracy'].idxmax(), 'layer']
    best_acc = data['accuracy'].max()
    report.append(f"  {probe_type.upper()} Probe:")
    report.append(f"    Best layer: {best_layer} (accuracy: {best_acc:.3f})")
    report.append(f"    All layers: {data[['layer', 'accuracy']].to_dict('records')}")
report.append("")

# WHAT results
report.append("WHAT: INTERVENTION ANALYSIS RESULTS")
report.append("-" * 80)
for probe_type in ['linear', 'mlp']:
    report.append(f"\n  {probe_type.upper()} Probe:")
    data = accuracy_drops[accuracy_drops['probe_type'] == probe_type]

    for intervention in ['reversed', 'shuffled']:
        subset = data[data['intervention'] == intervention]
        avg_drop = subset['drop_absolute'].mean()
        max_drop = subset['drop_absolute'].max()
        max_drop_layer = subset.loc[subset['drop_absolute'].idxmax(), 'layer']

        report.append(f"    {intervention.capitalize()}:")
        report.append(f"      Average accuracy drop: {avg_drop:.3f}")
        report.append(f"      Max drop: {max_drop:.3f} at layer {max_drop_layer}")

report.append("")
report.append("="*80)
report.append("KEY FINDINGS:")
report.append("="*80)

# Auto-generate findings
best_linear_layer = results_df[results_df['probe_type'] == 'linear']['accuracy'].idxmax()
best_linear_acc = results_df.loc[best_linear_layer, 'accuracy']
best_linear_layer_idx = results_df.loc[best_linear_layer, 'layer']

report.append(f"1. Temporal information is most strongly encoded in layer {best_linear_layer_idx}")
report.append(f"   (linear probe accuracy: {best_linear_acc:.3f})")

# Check if reverse hurts more than shuffle
avg_reverse_drop = accuracy_drops[accuracy_drops['intervention'] == 'reversed']['drop_absolute'].mean()
avg_shuffle_drop = accuracy_drops[accuracy_drops['intervention'] == 'shuffled']['drop_absolute'].mean()

if avg_reverse_drop > avg_shuffle_drop:
    report.append("\n2. Model DOES rely on temporal order (reverse hurts more than shuffle)")
    report.append(f"   Average reverse drop: {avg_reverse_drop:.3f}")
    report.append(f"   Average shuffle drop: {avg_shuffle_drop:.3f}")
else:
    report.append("\n2. Model MAY use shortcuts (shuffle hurts more than reverse)")
    report.append(f"   Average reverse drop: {avg_reverse_drop:.3f}")
    report.append(f"   Average shuffle drop: {avg_shuffle_drop:.3f}")

# MLP vs Linear comparison
mlp_best = results_df[results_df['probe_type'] == 'mlp']['accuracy'].max()
linear_best = results_df[results_df['probe_type'] == 'linear']['accuracy'].max()

if mlp_best > linear_best + 0.05:
    report.append(f"\n3. Non-linear probe performs significantly better than linear")
    report.append(f"   MLP best: {mlp_best:.3f}, Linear best: {linear_best:.3f}")
    report.append("   This suggests features require non-linear transformation")
else:
    report.append(f"\n3. Linear probe performs comparably to non-linear")
    report.append(f"   MLP best: {mlp_best:.3f}, Linear best: {linear_best:.3f}")
    report.append("   This suggests features are already linearly separable")

report.append("")
report.append("="*80)

# Print and save report
report_text = "\n".join(report)
print(report_text)

with open(f"{RESULTS_DIR}/summary_report.txt", 'w') as f:
    f.write(report_text)

print(f"\n Report saved: {RESULTS_DIR}/summary_report.txt")

### Next experiments to do:

- **Task-specific analysis:** Compare directional vs state transition vs terminal completion
- **Different models:** Trying VideoMAE-base, TimeSformer, or ViViT
- **More interventions:** Frame dropping, temporal jittering, etc.
- **Scale up:** Use full dataset instead of POC subset.